# Task 10: surrogate analyses on a Colab GPU

The pool replay, the online replay against the exact front, the two-level model against single-level,
and the inverse query of `docs/specs/task10_drone_sizing.md` section 10. **Requires T10.9 records
committed and T10.10 merged.** The GPs have at most 192 training points, so the GPU matters only for the
21^6 inverse grid (85.8 million predictions) and the online replay's candidate scoring; section 2 measures
it on this runtime and prints whether it helped, so the claim is recorded rather than assumed.

A GPU runtime (T4 is enough). The pool replay alone also runs on a CPU runtime in about half an hour.

## 0. Environment

In [ ]:
!nvidia-smi -L || echo "no GPU: the inverse grid and the online replay will run on the CPU"
import os, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "device", DEVICE, "cores", os.cpu_count())

## 1. Repository

In [ ]:
COMMIT = "main"   # branch, tag or commit sha; pin it to the commit that holds the task named above
import os, subprocess, sys
if not os.path.isdir("/content/ai-engineering"):
    !git clone -q https://github.com/emarinhoss/ai-engineering.git /content/ai-engineering
%cd /content/ai-engineering
stale = [m for m in sys.modules if m.split(".")[0] in ("physics", "geometry", "drone", "surrogate", "analysis", "designs")]
assert not stale, f"repo modules already imported in this kernel ({stale[:3]}...): Runtime -> Restart runtime, then run again"
!git fetch -q --all
!git checkout -q {COMMIT}
!git pull -q --ff-only
!pip install -q -e .
sys.path.insert(0, "/content/ai-engineering")
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("commit", HEAD)

## 2. Does the GPU help here?
Fits one GP on the 96-point high-fidelity pool (CPU, float64, as every recorded fit) and predicts one
million grid points on the CPU and on the GPU. Look at: the two times. The fit stays on the CPU in every
analysis below, because the recorded hyperparameters must be bit-identical to a container fit; only
prediction may move to the GPU, and only if this cell shows a gain.

In [ ]:
import time, numpy as np
from analysis.pareto import fit_gps, load_rows, predict
from designs.make_pool_drone import LOW, HIGH
rows = load_rows("runs/pool_af/results.csv")
models, theta, F, _ = fit_gps(rows, family="drone")
pts = np.random.default_rng(0).uniform(LOW, HIGH, size=(1_000_000, len(LOW)))
t0 = time.perf_counter(); predict(models, pts, family="drone"); t_cpu = time.perf_counter() - t0
t0 = time.perf_counter(); predict(models, pts, family="drone", device=DEVICE); t_dev = time.perf_counter() - t0
print(f"1e6 predictions: cpu {t_cpu:.1f} s, {DEVICE} {t_dev:.1f} s, speed-up {t_cpu / t_dev:.1f}x")
GPU_HELPS = DEVICE == "cuda" and t_cpu / t_dev > 2.0
print("GPU used for prediction below:", GPU_HELPS)

## 3. Pool replay (question 2, first measurement)
`surrogate.replay` with the drone bounds: random, LHS, max-variance, EHVI, 10 seeds, budgets 12 to 96 on
the 96-point SU2 pool. Writes `runs/replay_drone/<stamp>/replay.csv`. About 30 minutes on the CPU.

In [ ]:
import datetime, subprocess
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + HEAD[:7]
OUT = f"/content/drive/MyDrive/su2queue/drone_analysis/{STAMP}"
from google.colab import drive; drive.mount("/content/drive"); os.makedirs(OUT, exist_ok=True)
!python -m surrogate.replay --family drone --pool designs/pool_drone96.csv --results runs/pool_af/results.csv --seeds 10 --n-max 96 --out {OUT}/replay.csv

## 4. Online replay against the exact front (question 2, second measurement)
The same four strategies, but each step draws 4096 fresh Sobol candidates, scores them on the GP, and
evaluates the chosen design with the low-fidelity model itself (no pool); regret is against
`runs/drone_grid/<date>/front.csv`. This is the measurement the inlet replay could not make.

In [ ]:
GRID = sorted(__import__("glob").glob("runs/drone_grid/*/front.csv"))[-1]
DEV = "--device cuda" if GPU_HELPS else ""
!python -m analysis.replay_drone_online --front {GRID} --seeds 10 --n-max 96 {DEV} --out {OUT}/replay_online.csv

## 5. Two-level against single-level (question 3)
`analysis.mf_drone`: held-out error against the number of SU2 designs, 10 seeds, budgets 8 to 96, with the 192-point low-fidelity pool.

In [ ]:
!python -m analysis.mf_drone --results runs/pool_af/results.csv --low runs/drone_grid/lf192.csv --seeds 10 --out {OUT}/mf.csv

## 6. Inverse query (question 4)
Longest endurance with P(mtow <= 4.0 kg) >= 0.9 on the 21^6 grid inside the gates; the cap is the
ledger's `mass_cap_kg`, fixed before any run. Writes `result.json` and `params.json` for the
confirmation run (five SU2 cases through the queue).

In [ ]:
!python -m analysis.inverse --family drone --results runs/pool_af/results.csv --grid 21 {DEV} --out {OUT}/inverse
print(open(f"{OUT}/inverse/result.json").read()[:1500])

## What to report back
The four output files (`replay.csv`, `replay_online.csv`, `mf.csv`, `inverse/result.json`) and the
GPU line of section 2. Claude copies them into `runs/` with their sha256 and commits; the figures and the
report are built from the committed copies, never from this notebook's output cells.